In [1]:
import torch

In [2]:
print("CUDA 사용 가능 여부:", torch.cuda.is_available())

CUDA 사용 가능 여부: True


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install librosa soundfile

Note: you may need to restart the kernel to use updated packages.


-------------------------------------------------------------------

In [17]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import load_dataset, Dataset, Audio
from tqdm import tqdm
import pandas as pd
from typing import Any, Dict, List

# 전처리된 데이터셋 로드 (예: CSV 파일로 저장한 경우)
data = pd.read_csv("1202_data_part1.csv")  # 'audio_path', 'transcription' 열 포함
#data['audio_path'] = data['audio_path'].str.replace(
 #   "converted_wav_files", "new_converted_wav_files"
#)

# 데이터셋 생성 및 오디오 파일 로드 확인
dataset = Dataset.from_pandas(data)
dataset = dataset.cast_column("audio_path", Audio())  # 오디오 파일 로드

# Whisper 모델 및 프로세서 불러오기
model_name = "openai/whisper-base"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

# 데이터셋 전처리 함수
def preprocess_data(examples):
    try:
        audio = examples["audio_path"]

        # 오디오 데이터가 올바르게 로드되지 않았을 경우 건너뛰기
        if not audio or "array" not in audio or "sampling_rate" not in audio:
            print(f"오디오 파일이 제대로 로드되지 않았습니다. 건너뛰기: {examples['audio_path']}")
            return {"input_features": None, "labels": None}

        inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features
        labels = processor(text=examples["transcription"], return_tensors="pt").input_ids
        return {"input_features": inputs.squeeze(), "labels": labels.squeeze()}

    except Exception as e:
        print(f"오류 발생: {e}. 건너뛰기: {examples['audio_path']}")
        return {"input_features": None, "labels": None}

# tqdm을 사용해 데이터셋 전처리
print("데이터셋 전처리 중...")
dataset = dataset.map(preprocess_data, remove_columns=["audio_path", "transcription"])

# None 값이 포함된 데이터 필터링하여 제거
dataset = dataset.filter(lambda example: example["input_features"] is not None and example["labels"] is not None)

데이터셋 전처리 중...


Map:   0%|          | 0/62678 [00:00<?, ? examples/s]

오류 발생: You need to specify either an `audio` or `text` input to process.. 건너뛰기: {'path': 'C:\\Users\\MATH-1\\dat_nlp\\한영 혼합 인식 데이터\\01.데이터\\1.Training\\원천데이터_0825_add\\TS11\\converted_pcm_to_wav\\f1_1_21_f2_1_28_211129_0010395.wav', 'array': array([0.        , 0.        , 0.        , ..., 0.00079346, 0.00064087,
       0.00042725]), 'sampling_rate': 16000}


Filter:   0%|          | 0/62678 [00:00<?, ? examples/s]

In [19]:
TRAIN_SAVE_PATH = "C:/Users/MATH-1/dat_nlp/1202_train_dataset_part1"
EVAL_SAVE_PATH = "C:/Users/MATH-1/dat_nlp/1202_eval_dataset_part1"
TEST_SPLIT = 0.2
RANDOM_SEED = 42

# 데이터셋 분리
train_test_split = dataset.train_test_split(test_size=TEST_SPLIT, seed=RANDOM_SEED)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

# 전처리된 데이터 저장
train_dataset.save_to_disk(TRAIN_SAVE_PATH)
eval_dataset.save_to_disk(EVAL_SAVE_PATH)
print("전처리된 데이터셋 저장 완료")

Saving the dataset (0/97 shards):   0%|          | 0/50141 [00:00<?, ? examples/s]

Saving the dataset (0/25 shards):   0%|          | 0/12536 [00:00<?, ? examples/s]

전처리된 데이터셋 저장 완료


## LoRA Finetuning

In [23]:
from datasets import Dataset, concatenate_datasets
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
import torch
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import os

In [25]:
# 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"

In [27]:
# 설정
train_data_dir = "C:/Users/MATH-1/dat_nlp/1202_train_dataset_part1"
model_name = "openai/whisper-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 데이터 로드 및 결합
def load_arrow_files(directory):
    dataset_list = []
    for file in tqdm(os.listdir(directory), desc="데이터 로드 중"):
        if file.endswith(".arrow"):
            file_path = os.path.join(directory, file)
            try:
                dataset = Dataset.from_file(file_path)
                dataset_list.append(dataset)
            except Exception as e:
                print(f"파일 로드 실패: {file_path}, 오류: {e}")
    if dataset_list:
        return concatenate_datasets(dataset_list)
    else:
        raise ValueError("유효한 데이터셋 파일이 없습니다.")

print("데이터 로드 중...")
train_dataset = load_arrow_files(train_data_dir)

# 필터링 전에 데이터셋 내용 확인
print(f"데이터셋 크기(필터링 전): {len(train_dataset)}")
print(f"샘플 데이터 확인: {train_dataset[0]}")  # 첫 번째 샘플 확인

# None 값 필터링
train_dataset = train_dataset.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)

# 필터링 후 데이터셋 크기 확인
print(f"데이터셋 크기(필터링 후): {len(train_dataset)}")
if len(train_dataset) == 0:
    raise ValueError("필터링 결과 데이터셋이 비어 있습니다. 데이터를 확인하세요.")

데이터 로드 중...


데이터 로드 중: 100%|██████████████████████████████████████████████████████████████████| 99/99 [00:09<00:00,  9.99it/s]
IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Filter:   0%|          | 0/50141 [00:00<?, ? examples/s]

데이터셋 크기(필터링 후): 50141


In [ ]:
from torch.nn.utils.rnn import pad_sequence

# Whisper 모델 및 프로세서 준비
processor = WhisperProcessor.from_pretrained(model_name)

# LoRA 설정
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    #task_type="SEQ_2_SEQ_LM"
)

# Whisper 모델 로드 및 LoRA 적용
print("모델 준비 중...")
model = WhisperForConditionalGeneration.from_pretrained(model_name)
model = get_peft_model(model, lora_config)  # LoRA 적용
model.to(device)

# DataLoader 생성
MAX_LABEL_LENGTH = 60  # 고정된 최대 길이

def collate_fn(batch):
    input_features = [torch.tensor(item["input_features"]) for item in batch]
    labels = [torch.tensor(item["labels"]) for item in batch]

    # 입력 데이터 패딩
    input_features = pad_sequence(input_features, batch_first=True, padding_value=0.0)

    # 라벨 패딩 - 고정된 MAX_LABEL_LENGTH 적용
    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id
    )

    # 라벨 길이를 고정된 MAX_LABEL_LENGTH로 자르거나 패딩
    if labels.shape[1] < MAX_LABEL_LENGTH:
        padding = torch.full(
            (labels.shape[0], MAX_LABEL_LENGTH - labels.shape[1]),
            processor.tokenizer.pad_token_id,
        )
        labels = torch.cat([labels, padding], dim=1)
    else:
        labels = labels[:, :MAX_LABEL_LENGTH]

    return {"input_features": input_features, "labels": labels}


train_dataloader = DataLoader(train_dataset, batch_size=8, collate_fn=collate_fn, shuffle=True)

# 모델 학습 설정
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()

# 모델 학습 루프
print("모델 학습 시작...")
for epoch in range(3):
    print(f"\nEpoch {epoch + 1} 시작...")
    epoch_loss = 0

    with tqdm(train_dataloader, desc=f"Epoch {epoch + 1} 진행 중", leave=True, dynamic_ncols=True) as progress_bar:
        for batch in progress_bar:
            # 배치 데이터 추출
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            # Whisper 모델 호출
            outputs = model(input_features=input_features, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})  # Loss 표시

    avg_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1} 완료, 평균 Loss: {avg_loss:.4f}")

# 모델 저장
output_dir = "1202_lora_whisper_model"
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

print(f"모델 저장 완료: {output_dir}")


모델 준비 중...
모델 학습 시작...

Epoch 1 시작...


Epoch 1 진행 중: 100%|██████████████████████████████████████████████| 6268/6268 [3:01:27<00:00,  1.74s/it, Loss=1.4767]


Epoch 1 완료, 평균 Loss: 1.9206

Epoch 2 시작...


Epoch 2 진행 중:  30%|█████████████▋                                | 1857/6268 [57:09<2:09:01,  1.75s/it, Loss=1.7593]

In [ ]:
------------------------------------------------------------

In [17]:
# 모델 파라미터가 GPU에 있는지 확인
if torch.cuda.is_available():
    is_on_gpu = all(param.is_cuda for param in model.parameters())
    if is_on_gpu:
        print("모델이 GPU에 성공적으로 적용되었습니다.")
    else:
        print("모델이 GPU에 적용되지 않았습니다. CPU에서 실행 중입니다.")
else:
    print("GPU가 사용 가능하지 않습니다. CPU에서 실행 중입니다.")

모델이 GPU에 성공적으로 적용되었습니다.


In [ ]:
import zipfile
import os
import json
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset, Audio
from tqdm import tqdm
import gc

# 데이터 폴더와 결과 저장 위치 설정
data_folder = 'C:/Users/MATH-1/dat_nlp/005.한영 혼합 인식 데이터/01.데이터/1.Training/라벨링데이터_0825_add'
wav_folder = 'C:/Users/MATH-1/dat_nlp/converted_wav_files'
output_data = []

# zip 파일당 최대 300개의 JSON 파일만 처리
for zip_file in sorted(os.listdir(data_folder)):
    zip_path = os.path.join(data_folder, zip_file)
    
    if zipfile.is_zipfile(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            json_count = 0
            for file in z.namelist():
                if file.endswith('.json') and json_count < 300:
                    with z.open(file) as f:
                        data = json.load(f)
                        dialogs = data.get("dialogs", [])
                        audio_file_name = os.path.basename(file).replace(".json", ".wav")
                        audio_path = os.path.join(wav_folder, audio_file_name)

                        for dialog in dialogs:
                            if "deleted" not in dialog:
                                text = dialog["text"]
                                for expression in dialog.get("expression", []):
                                    text = text.replace(expression["form"], expression["originalForm"])
                                output_data.append({
                                    "audio_path": audio_path,
                                    "transcription": text,
                                    "start_time": float(dialog["startTime"]),
                                    "end_time": float(dialog["endTime"])
                                })
                    json_count += 1

# DataFrame으로 정리 후 CSV로 저장
df = pd.DataFrame(output_data)
df.to_csv("whisper_finetuning_data.csv", index=False, encoding='utf-8-sig')

# 데이터셋 생성 및 전처리
data = pd.read_csv("whisper_finetuning_data.csv")
dataset = Dataset.from_pandas(data).cast_column("audio_path", Audio())

# Whisper 모델 및 프로세서 불러오기
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

# 전처리 함수
def preprocess_data(examples):
    try:
        audio = examples["audio_path"]
        if not audio or "array" not in audio or "sampling_rate" not in audio:
            print(f"오디오 파일이 로드되지 않았습니다: {examples['audio_path']}")
            return {"input_features": None, "labels": None}
        
        inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features
        labels = processor(text=examples["transcription"], return_tensors="pt").input_ids
        return {"input_features": inputs.squeeze(), "labels": labels.squeeze()}
    except Exception as e:
        print(f"오류 발생: {e}, 파일 건너뜀: {examples['audio_path']}")
        return {"input_features": None, "labels": None}

# 데이터셋 전처리 및 메모리 효율성 개선
print("데이터셋 전처리 중...")
dataset = dataset.map(preprocess_data, remove_columns=["audio_path", "transcription"], keep_in_memory=False, load_from_cache_file=False)
dataset = dataset.filter(
    lambda example: example["input_features"] is not None and example["labels"] is not None,
    writer_batch_size=8  # 메모리 효율 개선을 위해 설정
)

# 메모리 정리
gc.collect()

# 학습 파라미터 설정 (배치 크기 줄임)
training_args = TrainingArguments(
    output_dir="./whisper_finetuning",
    per_device_train_batch_size=2,
    evaluation_strategy="epoch",
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
)

# Trainer 설정 및 학습 시작
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
)

print("Whisper 모델 파인튜닝 시작...")
trainer.train()
model.save_pretrained("./whisper_finetuned_model")
print("모델 파인튜닝 완료 및 저장 완료")
